# Lab: Nhận diện khuôn mat thoi gian thuc voi FaceNet + MTCNN

### Thông tin nhóm thực hiện:

| STT | Họ và Tên | MSSV | Vai trò/Đánh giá |
|-----|-----------|------|------------------|
| 1 | **Huỳnh Thế Hy** | 051205009083 | Nhóm trưởng |
| 2 | Nghiêm Đức Thuận | 060205003756 | Thành viên |
| 3 | Đào Thiện Nhân | 052205008343 | Thành viên |
| 4 | Tạ Nguyễn Quốc Triệu | 051205004949 | Thành viên |
| 5 | Hoàng Phú | 2251120373 | Thành viên |
| 6 | Thanh Tân | | Thành viên |

## Mo ta tong quan

Bai lab nay xay dung he thong nhan dien khuon mat thoi gian thuc su dung webcam. Pipeline xu ly:

```
Webcam --> Detect Face (MTCNN) --> Crop Face --> Preprocess --> FaceNet Embedding --> Compare Similarity --> Hien thi ket qua
```

Nguong quyet dinh (threshold = 0.7):
- similarity > 0.7  --> Hien ten nguoi dung (Matched)
- similarity <= 0.7 --> Hien Unknown



---

## Cơ sở lý thuyết

### 1. FaceNet la gi?

FaceNet la mo hinh deep learning do Google phat trien (Schroff et al., 2015). No hoat dong theo nguyen tac:

- **Input:** Anh khuon mat da duoc crop va resize ve 160x160 pixel
- **Output:** Vector embedding 128 chieu (hoac 512 chieu tuy phien ban)
- **Y tuong chinh:** Nhung khuon mat cua cung mot nguoi se co vector embedding nam gan nhau trong khong gian 128 chieu. Nguoi khac nhau se co vector xa nhau.
- **Ham loss:** FaceNet duoc huan luyen bang Triplet Loss:
  - Anchor: Anh goc
  - Positive: Anh cung nguoi voi Anchor
  - Negative: Anh nguoi khac
  - Muc tieu: `d(Anchor, Positive) + margin < d(Anchor, Negative)`

Uu diem cua FaceNet:
- Khong can huan luyen lai khi them nguoi moi (chi can them embedding)
- Do chinh xac cao tren nhieu tap du lieu benchmark
- Dung luong mo hinh vua phai

---

### 2. MTCNN la gi?

MTCNN (Multi-task Cascaded Convolutional Networks) la mot trong nhung giai thuat detect khuon mat manh nhat hien nay. Kien truc gom 3 tang:

- **P-Net (Proposal Network):** Quyet vu quet toan bo anh o nhieu ti le, de xuat cac vung co the la khuon mat
- **R-Net (Refine Network):** Loc bot cac proposal sai, chinh xac hoa bounding box
- **O-Net (Output Network):** Xac nhan cuoi cung, tra ve bounding box chinh xac va 5 landmark (mat trai, mat phai, mui, goc trai mieng, goc phai mieng)

MTCNN hoat dong tot voi:
- Khuon mat nhieu goc nghieng
- Anh sang khong hoan hao
- Nhieu khuon mat trong mot khung hinh

---

### 3. Vi sao ket hop MTCNN + FaceNet?

| Nhiem vu | Mo hinh | Chi tiet |
|----------|---------|----------|
| Phat hien khuon mat | MTCNN | Xac dinh vi tri khuon mat trong frame |
| Nhan dien khuon mat | FaceNet | So sanh embedding voi database |

Hai mo hinh bo tro nhau hoan hao:
- MTCNN cho biet khuon mat o dau trong anh
- FaceNet cho biet khuon mat do la ai

### 4. Cosine Similarity la gi?

Do do tuong dong giua hai vector dua tren goc giua chung:

```
cosine_similarity(A, B) = (A . B) / (||A|| * ||B||)
```

- Ket qua tu -1 den 1
- Gan 1 = rat giong nhau
- Gan 0 = khong lien quan
- Gan -1 = trai nguoc

Trong nhan dien khuon mat, ta dung nguong 0.7: neu similarity >= 0.7 thi coi la cung nguoi.

---

## Cell 3 - Cai dat thu vien

Chay cell nay mot lan dau de cai tat ca dependency can thiet.  
Neu da cai roi thi co the bo qua.

**Luu y version:**
- `keras-facenet` can tensorflow >= 2.x
- `mtcnn` tuong thich tot voi tensorflow 2.x
- Neu gap loi conflict, chay: `pip install tensorflow==2.13.0`

In [1]:
# Cai dat cac thu vien can thiet
# Chi can chay mot lan dau tien

import subprocess
import sys

def install_package(package):
    """Cai dat package va bat loi neu that bai."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package, "-q"]
        )
        print(f"[OK] {package} da cai thanh cong")
    except subprocess.CalledProcessError:
        print(f"[FAIL] Khong the cai {package} - kiem tra lai pip hoac ket noi mang")

packages = [
    "opencv-python",
    "mtcnn",
    "keras-facenet",
    "scikit-learn",
    "Pillow",
    "numpy",
    "matplotlib",
]

print("Bat dau cai dat cac thu vien...")
for pkg in packages:
    install_package(pkg)

print("\nHoan tat cai dat. Hay restart kernel neu gap loi import sau do.")

Bat dau cai dat cac thu vien...
[OK] opencv-python da cai thanh cong
[OK] mtcnn da cai thanh cong
[OK] keras-facenet da cai thanh cong
[OK] scikit-learn da cai thanh cong
[OK] Pillow da cai thanh cong
[OK] numpy da cai thanh cong
[OK] matplotlib da cai thanh cong

Hoan tat cai dat. Hay restart kernel neu gap loi import sau do.


---

## Cell 4 - Import thu vien

Import tat ca cac package can thiet. Cell nay se:
- Kiem tra xem tung thu vien co import duoc khong
- Hien thi version de tien debug sau nay
- Neu import loi, in huong dan sua cu the

**Loi thuong gap:**
- `DLL load failed` (Windows): Cai lai Microsoft Visual C++ Redistributable
- `No module named 'mtcnn'`: Chay lai Cell 3
- `keras_facenet not found`: Thu `pip install keras-facenet --upgrade`

In [ ]:
# # ============================================================
# # IMPORT PACKAGES
# # ============================================================

# import os
# import sys
# import time
# import warnings
# import csv
# from datetime import datetime
# from pathlib import Path

# warnings.filterwarnings("ignore")

# # Computer Vision
# try:
#     import cv2
#     print(f"[OK] opencv-python version: {cv2.__version__}")
# except ImportError:
#     print("[FAIL] OpenCV chua duoc cai. Chay: pip install opencv-python")
#     sys.exit(1)

# # Numerical computing
# try:
#     import numpy as np
#     print(f"[OK] numpy version: {np.__version__}")
# except ImportError:
#     print("[FAIL] numpy chua duoc cai. Chay: pip install numpy")
#     sys.exit(1)

# # Image processing
# try:
#     from PIL import Image
#     import PIL
#     print(f"[OK] Pillow version: {PIL.__version__}")
# except ImportError:
#     print("[FAIL] Pillow chua duoc cai. Chay: pip install Pillow")

# # Visualization
# try:
#     import matplotlib
#     import matplotlib.pyplot as plt
#     print(f"[OK] matplotlib version: {matplotlib.__version__}")
# except ImportError:
#     print("[FAIL] matplotlib chua duoc cai. Chay: pip install matplotlib")

# # Machine Learning utilities
# try:
#     from sklearn.metrics.pairwise import cosine_similarity
#     import sklearn
#     print(f"[OK] scikit-learn version: {sklearn.__version__}")
# except ImportError:
#     print("[FAIL] scikit-learn chua duoc cai. Chay: pip install scikit-learn")

# # Face detection
# try:
#     from mtcnn import MTCNN
#     print("[OK] MTCNN import thanh cong")
# except ImportError:
#     print("[FAIL] MTCNN chua duoc cai. Chay: pip install mtcnn")
#     sys.exit(1)

# # Face embedding
# try:
#     from keras_facenet import FaceNet
#     print("[OK] keras-facenet import thanh cong")
# except ImportError:
#     print("[FAIL] keras-facenet chua duoc cai. Chay: pip install keras-facenet")
#     sys.exit(1)

# print("\nTat ca thu vien da san sang.")

[OK] opencv-python version: 4.8.1
[OK] numpy version: 1.26.4
[OK] Pillow version: 12.2.0
[OK] matplotlib version: 3.8.4
[OK] scikit-learn version: 1.8.0
[OK] MTCNN import thanh cong
[OK] keras-facenet import thanh cong

Tat ca thu vien da san sang.


---

## Cell 5 - Khoi tao mo hinh

Load MTCNN va FaceNet vao RAM. Qua trinh nay mat khoang 10-30 giay lan dau (phai download weights).

**Giai thich ky thuat:**
- `MTCNN()`: Khoi tao detector, load weights cua 3 mang P-Net, R-Net, O-Net
- `FaceNet()`: Khoi tao embedder, load weights pretrained tren hang trieu anh khuon mat
- Ca hai model deu su dung TensorFlow backend

**Loi thuong gap:**
- RAM thieu: Dong cac tab trinh duyet truoc khi chay
- GPU khong phat hien: Binh thuong, model van chay tren CPU
- Loi DLL tren Windows: Cai Microsoft Visual C++ 2019 Redistributable

In [3]:
# ============================================================
# KHOI TAO MODEL
# ============================================================

print("Dang khoi tao MTCNN detector...")
t0 = time.time()
detector = MTCNN()
print(f"MTCNN san sang sau {time.time() - t0:.2f} giay")

print("Dang khoi tao FaceNet embedder (co the mat vai phut lan dau)...")
t0 = time.time()
embedder = FaceNet()
print(f"FaceNet san sang sau {time.time() - t0:.2f} giay")

print("\nCa hai mo hinh da san sang su dung.")

Dang khoi tao MTCNN detector...
MTCNN san sang sau 0.49 giay
Dang khoi tao FaceNet embedder (co the mat vai phut lan dau)...

FaceNet san sang sau 56.66 giay

Ca hai mo hinh da san sang su dung.


---

## Cell 6 - Kiem tra webcam

Truoc khi lam gi, kiem tra webcam co hoat dong khong. Day la buoc debug quan trong de phat hien som cac van de:
- Camera index sai (thu 0, 1, 2...)
- App khac dang chiem camera (Zoom, Teams...)
- Driver camera chua cai

**Input:** Khong co  
**Output:** Thong bao camera co hoat dong hay khong

In [4]:
# ============================================================
# KIEM TRA WEBCAM
# ============================================================

def check_webcam(camera_index: int = 0) -> bool:
    """
    Kiem tra webcam co hoat dong khong.
    
    Args:
        camera_index: Index cua camera (0 = cam chinh, 1 = cam phu)
    
    Returns:
        True neu cam OK, False neu co van de
    """
    print(f"Dang kiem tra camera index {camera_index}...")
    
    cap = cv2.VideoCapture(camera_index)
    
    if not cap.isOpened():
        print(f"[FAIL] Khong the mo camera {camera_index}")
        print("Huong dan sua loi:")
        print("  1. Kiem tra camera co cam vao may tinh khong")
        print("  2. Dong tat ca app dang dung camera (Zoom, Teams, Skype)")
        print("  3. Thu doi camera_index tu 0 sang 1 hoac 2")
        print("  4. Kiem tra Device Manager (Windows) xem camera co bi disable")
        cap.release()
        return False
    
    ret, frame = cap.read()
    
    if not ret or frame is None:
        print("[FAIL] Camera mo duoc nhung khong doc duoc frame")
        print("Thu khoi dong lai may hoac cap nhat driver camera")
        cap.release()
        return False
    
    height, width = frame.shape[:2]
    print(f"[OK] Camera {camera_index} hoat dong tot")
    print(f"     Resolution: {width}x{height}")
    print(f"     Frame shape: {frame.shape}")
    
    cap.release()
    return True


# Thay doi index o day neu camera chinh khong hoat dong
CAMERA_INDEX = 0

camera_ok = check_webcam(CAMERA_INDEX)

if not camera_ok:
    # Thu tim camera khac
    print("\nDang thu tim camera khac...")
    for idx in range(1, 4):
        if check_webcam(idx):
            CAMERA_INDEX = idx
            camera_ok = True
            print(f"Tim thay camera tai index {idx}")
            break

if not camera_ok:
    print("\nKhong tim thay camera nao. Vui long kiem tra lai thiet bi.")

Dang kiem tra camera index 0...
[OK] Camera 0 hoat dong tot
     Resolution: 640x480
     Frame shape: (480, 640, 3)


---

## Cell 7 - Dinh nghia cac ham xu ly khuon mat

Day la phan core cua toan bo he thong. Bao gom 3 ham chinh:

1. `extract_face(image)`: Dung MTCNN detect khuon mat, crop va resize ve 160x160
2. `get_embedding(face_pixels)`: Dung FaceNet tao vector 512 chieu
3. `compare_faces(embedding, database)`: So sanh embedding voi toan bo database

**Tai sao viet thanh ham rieng?**
- De test doc lap tung buoc
- De tai su dung (reuse) o nhieu noi
- De doc da va bao tri
- Tien debug khi co loi

**Loi thuong gap trong extract_face:**
- Anh toi: MTCNN khong detect duoc, tra ve None
- Khuon mat qua nho trong frame: Tang do phan giai hoac dung camera o gan hon
- Bounding box bi out of bound: Can clip toa do ve gioi han anh

In [5]:
# ============================================================
# CAC HAM XU LY KHUON MAT (CORE FUNCTIONS)
# ============================================================

# Kich thuoc chuan ma FaceNet yeu cau
FACE_SIZE = 160  # pixels


def extract_face(
    image: np.ndarray,
    required_size: tuple = (FACE_SIZE, FACE_SIZE)
) -> np.ndarray | None:
    """
    Detect va crop khuon mat dau tien trong anh.
    
    Args:
        image: Anh BGR (tu OpenCV) hoac RGB
        required_size: Kich thuoc output (chieu rong, chieu cao)
    
    Returns:
        numpy array anh khuon mat da resize (H, W, 3), hoac None neu khong detect duoc
    """
    # MTCNN can anh RGB, OpenCV doc anh BGR -> convert
    if len(image.shape) == 3 and image.shape[2] == 3:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = image
    
    # Detect khuon mat
    results = detector.detect_faces(image_rgb)
    
    if not results:
        return None  # Khong tim thay khuon mat nao
    
    # Lay khuon mat co do tin cay cao nhat
    best_face = max(results, key=lambda x: x['confidence'])
    
    if best_face['confidence'] < 0.9:
        return None  # Detect nhung do tin cay qua thap
    
    # Lay toa do bounding box
    x1, y1, width, height = best_face['box']
    
    # Clip toa do ve gioi han anh (tranh negative index)
    img_h, img_w = image_rgb.shape[:2]
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img_w, x1 + width)
    y2 = min(img_h, y1 + height)
    
    # Kiem tra vung crop hop le
    if x2 <= x1 or y2 <= y1:
        return None
    
    # Crop khuon mat
    face_pixels = image_rgb[y1:y2, x1:x2]
    
    # Resize ve kich thuoc chuan
    face_image = Image.fromarray(face_pixels)
    face_image = face_image.resize(required_size)
    face_array = np.array(face_image)
    
    return face_array


def extract_face_with_box(
    image: np.ndarray,
    required_size: tuple = (FACE_SIZE, FACE_SIZE)
) -> tuple:
    """
    Detect khuon mat va tra ve ca face array lan bounding box.
    Can thiet cho realtime display (ve box len frame).
    
    Returns:
        Tuple (face_array, bounding_box_dict) hoac (None, None)
    """
    if len(image.shape) == 3 and image.shape[2] == 3:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = image
    
    results = detector.detect_faces(image_rgb)
    
    if not results:
        return None, None
    
    best_face = max(results, key=lambda x: x['confidence'])
    
    if best_face['confidence'] < 0.9:
        return None, None
    
    x1, y1, width, height = best_face['box']
    img_h, img_w = image_rgb.shape[:2]
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img_w, x1 + width)
    y2 = min(img_h, y1 + height)
    
    if x2 <= x1 or y2 <= y1:
        return None, None
    
    face_pixels = image_rgb[y1:y2, x1:x2]
    face_image = Image.fromarray(face_pixels)
    face_image = face_image.resize(required_size)
    face_array = np.array(face_image)
    
    box = {"x1": x1, "y1": y1, "x2": x2, "y2": y2, "confidence": best_face['confidence']}
    
    return face_array, box


def extract_all_faces_with_boxes(
    image: np.ndarray,
    required_size: tuple = (FACE_SIZE, FACE_SIZE),
    min_confidence: float = 0.9
) -> list:
    """
    Detect TAT CA khuon mat trong anh (cho phep nhan nhieu nguoi cung luc).
    
    Returns:
        List cac tuple (face_array, box_dict)
    """
    if len(image.shape) == 3 and image.shape[2] == 3:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = image
    
    results = detector.detect_faces(image_rgb)
    
    if not results:
        return []
    
    faces = []
    img_h, img_w = image_rgb.shape[:2]
    
    for face_data in results:
        if face_data['confidence'] < min_confidence:
            continue
        
        x1, y1, width, height = face_data['box']
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_w, x1 + width)
        y2 = min(img_h, y1 + height)
        
        if x2 <= x1 or y2 <= y1:
            continue
        
        face_pixels = image_rgb[y1:y2, x1:x2]
        face_image = Image.fromarray(face_pixels)
        face_image = face_image.resize(required_size)
        face_array = np.array(face_image)
        
        box = {
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "confidence": face_data['confidence']
        }
        faces.append((face_array, box))
    
    return faces


def get_embedding(face_pixels: np.ndarray) -> np.ndarray:
    """
    Trich xuat embedding vector 512 chieu tu anh khuon mat.
    
    Args:
        face_pixels: Anh khuon mat RGB, shape (160, 160, 3), dtype uint8
    
    Returns:
        Vector embedding shape (512,), dtype float32
    """
    # FaceNet can input la batch -> them 1 chieu
    # Shape: (160, 160, 3) -> (1, 160, 160, 3)
    face_batch = np.expand_dims(face_pixels, axis=0)
    
    # Tao embedding
    signature = embedder.embeddings(face_batch)
    
    # Lay ket qua cua anh dau tien trong batch
    return signature[0]


def compare_with_database(
    embedding: np.ndarray,
    database: dict,
    threshold: float = 0.7
) -> tuple:
    """
    So sanh embedding voi toan bo database de nhan dien nguoi.
    
    Args:
        embedding: Vector 512D cua khuon mat can nhan dien
        database: Dict {ten_nguoi: mean_embedding_vector}
        threshold: Nguong similarity toi thieu de xem la matched
    
    Returns:
        Tuple (ten_nguoi, similarity_score)
        Ten la 'Unknown' neu khong match
    """
    if not database:
        return "Unknown", 0.0
    
    best_name = "Unknown"
    best_score = -1.0
    
    # Query vector can phai la 2D array cho cosine_similarity
    query = embedding.reshape(1, -1)
    
    for name, ref_embedding in database.items():
        ref = ref_embedding.reshape(1, -1)
        score = cosine_similarity(query, ref)[0][0]
        
        if score > best_score:
            best_score = score
            best_name = name
    
    # Ap dung threshold
    if best_score < threshold:
        return "Unknown", best_score
    
    return best_name, best_score


print("Tat ca ham xu ly khuon mat da dinh nghia thanh cong.")
print("Cac ham co san:")
print("  - extract_face(image)")
print("  - extract_face_with_box(image)")
print("  - extract_all_faces_with_boxes(image)")
print("  - get_embedding(face_pixels)")
print("  - compare_with_database(embedding, database, threshold)")

Tat ca ham xu ly khuon mat da dinh nghia thanh cong.
Cac ham co san:
  - extract_face(image)
  - extract_face_with_box(image)
  - extract_all_faces_with_boxes(image)
  - get_embedding(face_pixels)
  - compare_with_database(embedding, database, threshold)


---

## Cell 8 - Tao thu muc dataset va giao dien dang ky khuon mat

Trong thuc te, ban co 2 cach xay dung database:

**Cach 1 (thu cong):** Chup anh truoc, dat vao thu muc `dataset/ten_nguoi/`, chay code doc folder  
**Cach 2 (realtime):** Nhan phim `S` trong khi chay webcam de chup va luu truc tiep

Cell nay se:
1. Tao cau truc thu muc neu chua co
2. Huong dan sinh vien cach chup anh mau
3. Kiem tra dataset co du anh chua

**Quan trong:** Nen co toi thieu 3-5 anh/nguoi de embedding trung binh chinh xac hon

In [23]:
# ============================================================
# TAO DATASET STRUCTURE
# ============================================================

DATASET_DIR = Path("dataset")
ATTENDANCE_FILE = Path("attendance_log.csv")

def create_dataset_structure():
    """Tao thu muc dataset va cac nguoi mau."""
    DATASET_DIR.mkdir(exist_ok=True)
    print(f"Thu muc dataset: {DATASET_DIR.absolute()}")
    
    # Kiem tra cac nguoi da co trong dataset
    existing_people = [
        d.name for d in DATASET_DIR.iterdir()
        if d.is_dir() and not d.name.startswith('.')
    ]
    
    if existing_people:
        print(f"\nDa co {len(existing_people)} nguoi trong dataset:")
        for person in sorted(existing_people):
            person_dir = DATASET_DIR / person
            img_files = list(person_dir.glob('*.jpg')) + \
                        list(person_dir.glob('*.jpeg')) + \
                        list(person_dir.glob('*.png'))
            print(f"  - {person}: {len(img_files)} anh")
    else:
        print("\nDataset dang trong. Hay them anh theo huong dan sau:")
    
    print("\nCach them anh vao dataset:")
    print(f"  1. Tao thu muc: {DATASET_DIR}/TenNguoi/")
    print(f"  2. Them anh JPG vao thu muc do (it nhat 3 anh, khac goc mat)")
    print(f"  3. Hoac su dung tinh nang 'Nhan phim S' trong webcam loop")
    
    return existing_people


def create_sample_people():
    """
    Tao thu muc mau cho cac nguoi demo.
    Sinh vien can tu them anh vao cac thu muc nay.
    """
    sample_people = ["An", "Binh", "Chi"]
    for person in sample_people:
        person_dir = DATASET_DIR / person
        person_dir.mkdir(exist_ok=True)
    
    print("Da tao thu muc mau cho:", ", ".join(sample_people))
    print(f"\nBay gio hay them anh vao cac thu muc:")
    for person in sample_people:
        print(f"  dataset/{person}/  <-- Them 3-5 anh khuon mat {person} vao day")


# Chay khoi tao
existing = create_dataset_structure()

if not existing:
    print("\nTao thu muc nguoi mau de sinh vien them anh...")
    create_sample_people()

Thu muc dataset: d:\Computer_Vision\computer-vision\notebooks\dataset

Da co 4 nguoi trong dataset:
  - An: 0 anh
  - Binh: 0 anh
  - Chi: 0 anh
  - Nghiem Duc Thuan: 1 anh

Cach them anh vao dataset:
  1. Tao thu muc: dataset/TenNguoi/
  2. Them anh JPG vao thu muc do (it nhat 3 anh, khac goc mat)
  3. Hoac su dung tinh nang 'Nhan phim S' trong webcam loop


---

## Cell 9 - Trich xuat embedding tu dataset va xay dung database

Khi da co anh trong thu muc dataset, cell nay se:

1. Duyet qua tung nguoi trong `dataset/`
2. Voi moi nguoi, doc tat ca cac anh
3. Detect va crop khuon mat tu moi anh
4. Trich xuat embedding vector 512D
5. Tinh TRUNG BINH cac embedding cua cung mot nguoi
6. Luu vao dict `database`

**Tai sao dung trung binh embedding?**  
Vi moi anh chup o goc khac nhau, anh sang khac nhau se cho embedding khac nhau. Trung binh nhieu anh se tao ra vector dai dien chuan xac hon cho nguoi do.

**Input:** Thu muc `dataset/`  
**Output:** `database = {"An": vector_512D, "Binh": vector_512D, ...}`

In [24]:
# ============================================================
# XAY DUNG FACE DATABASE TU DATASET
# ============================================================

def load_image_safe(image_path: Path) -> np.ndarray | None:
    """
    Doc anh an toan, xu ly cac truong hop loi.
    
    Returns:
        numpy array BGR hoac None neu loi
    """
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            # Thu doc bang Pillow neu OpenCV that bai (anh loi nhe)
            pil_img = Image.open(image_path).convert('RGB')
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        return img
    except Exception as e:
        print(f"    Loi doc anh {image_path.name}: {e}")
        return None


def build_face_database(dataset_dir: Path) -> dict:
    """
    Doc toan bo dataset va xay dung database embedding.
    
    Args:
        dataset_dir: Duong dan den thu muc dataset
    
    Returns:
        Dict {ten_nguoi: mean_embedding_vector_512D}
    """
    database = {}
    supported_formats = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
    
    if not dataset_dir.exists():
        print(f"Thu muc {dataset_dir} khong ton tai.")
        return database
    
    person_dirs = sorted([
        d for d in dataset_dir.iterdir()
        if d.is_dir() and not d.name.startswith('.')
    ])
    
    if not person_dirs:
        print("Khong tim thay thu muc nao trong dataset.")
        return database
    
    print(f"Tim thay {len(person_dirs)} nguoi trong dataset.")
    print("-" * 50)
    
    for person_dir in person_dirs:
        name = person_dir.name
        print(f"\nXu ly: {name}")
        
        # Tim tat ca file anh
        image_files = [
            f for f in person_dir.iterdir()
            if f.suffix.lower() in supported_formats
        ]
        
        if not image_files:
            print(f"  Khong co anh trong thu muc {name}/ - Bo qua")
            continue
        
        print(f"  Tim thay {len(image_files)} anh")
        
        embeddings = []
        
        for img_path in image_files:
            # Doc anh
            img = load_image_safe(img_path)
            if img is None:
                continue
            
            # Detect va crop khuon mat
            face = extract_face(img)
            if face is None:
                print(f"    Khong detect duoc mat trong: {img_path.name}")
                continue
            
            # Trich xuat embedding
            try:
                emb = get_embedding(face)
                embeddings.append(emb)
                print(f"    OK: {img_path.name} -> embedding shape: {emb.shape}")
            except Exception as e:
                print(f"    Loi embedding {img_path.name}: {e}")
                continue
        
        if not embeddings:
            print(f"  Khong co embedding hop le cho {name}. Bo qua nguoi nay.")
            continue
        
        # Tinh trung binh embedding cua tat ca anh cua nguoi nay
        mean_embedding = np.mean(embeddings, axis=0)
        database[name] = mean_embedding
        
        print(f"  Ket qua: {len(embeddings)}/{len(image_files)} anh dung hop le")
        print(f"  Mean embedding shape: {mean_embedding.shape}")
    
    print("\n" + "-" * 50)
    print(f"Database hoan chinh: {len(database)} nguoi")
    for name in database:
        print(f"  - {name}: vector {database[name].shape}")
    
    return database


# Xay dung database
print("Bat dau xay dung face database...\n")
database = build_face_database(DATASET_DIR)

if not database:
    print("\nCANH BAO: Database trong.")
    print("He thong van chay nhung tat ca khuon mat se hien 'Unknown'.")
    print("Hay them anh vao thu muc dataset/ roi chay lai cell nay.")

Bat dau xay dung face database...

Tim thay 4 nguoi trong dataset.
--------------------------------------------------

Xu ly: An
  Khong co anh trong thu muc An/ - Bo qua

Xu ly: Binh
  Khong co anh trong thu muc Binh/ - Bo qua

Xu ly: Chi
  Khong co anh trong thu muc Chi/ - Bo qua

Xu ly: Nghiem Duc Thuan
  Tim thay 1 anh
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
    OK: 1.jpg -> embedding shape: (512,)
  Ket qua: 1/1 anh dung hop le
  Mean embedding shape: (512,)

--------------------------------------------------
Database hoan chinh: 1 nguoi
  - Nghiem Duc Thuan: vector (512,)


---

## Cell 10 - Dinh nghia ham ve UI tren frame

Phan nay chua cac ham ve bounding box, ten va cac thong tin len frame webcam.

**Thiet ke UI:**
- Khung mau xanh la (GREEN) neu nhan dien duoc
- Khung mau do (RED) neu la Unknown
- Hien ten, similarity score va FPS goc tren trai
- Font chu ro rang, co nen mo de doc de trong moi anh sang

**Tai sao dung OpenCV de ve thay vi mat khac?**  
Vi OpenCV cho phep ve truc tiep len numpy array la frame video, khong can render trung gian. Nhanh va don gian.

In [25]:
# ============================================================
# CAC HAM VE UI LEN FRAME
# ============================================================

# Mau sac (BGR format - OpenCV)
COLOR_MATCHED   = (0, 220, 0)    # Xanh la
COLOR_UNKNOWN   = (0, 0, 220)    # Do
COLOR_TEXT_BG   = (30, 30, 30)   # Nen toi cho chu
COLOR_TEXT      = (255, 255, 255) # Chu trang
COLOR_FPS       = (0, 200, 255)  # Vang cam cho FPS
FONT            = cv2.FONT_HERSHEY_SIMPLEX


def draw_face_box(
    frame: np.ndarray,
    box: dict,
    name: str,
    score: float
) -> np.ndarray:
    """
    Ve bounding box va thong tin nhan dien len frame.
    
    Args:
        frame: Frame BGR tu webcam
        box: Dict {x1, y1, x2, y2}
        name: Ten nguoi duoc nhan dien
        score: Similarity score (0-1)
    
    Returns:
        Frame da duoc ve
    """
    x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
    
    is_matched = name != "Unknown"
    color = COLOR_MATCHED if is_matched else COLOR_UNKNOWN
    
    # Ve bounding box
    thickness = 2
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    
    # Tao nhan hien thi
    label = f"{name} ({score:.2f})"
    
    # Tinh toan vi tri va kich thuoc chu
    font_scale = 0.7
    font_thickness = 2
    (text_w, text_h), baseline = cv2.getTextSize(
        label, FONT, font_scale, font_thickness
    )
    
    # Ve nen cho chu (de doc hon)
    label_y = max(y1 - 10, text_h + 5)
    cv2.rectangle(
        frame,
        (x1, label_y - text_h - baseline - 4),
        (x1 + text_w + 4, label_y + baseline),
        color,
        cv2.FILLED
    )
    
    # Ve chu
    cv2.putText(
        frame, label,
        (x1 + 2, label_y - 2),
        FONT, font_scale,
        COLOR_TEXT, font_thickness
    )
    
    return frame


def draw_fps_and_info(frame: np.ndarray, fps: float, mode: str = "") -> np.ndarray:
    """
    Ve FPS va thong tin debug o goc tren trai.
    """
    h, w = frame.shape[:2]
    
    # FPS
    fps_text = f"FPS: {fps:.1f}"
    cv2.putText(frame, fps_text, (10, 30), FONT, 0.8, COLOR_FPS, 2)
    
    # Mode hoac huong dan
    if mode:
        cv2.putText(frame, mode, (10, 60), FONT, 0.55, COLOR_TEXT, 1)
    
    # Huong dan phim tat
    help_text = "[Q] Thoat  [S] Luu mat moi"
    cv2.putText(frame, help_text, (10, h - 15), FONT, 0.5, (180, 180, 180), 1)
    
    return frame


def draw_no_face_indicator(frame: np.ndarray) -> np.ndarray:
    """
    Hien thi thong bao khi khong detect duoc khuon mat.
    """
    h, w = frame.shape[:2]
    cv2.putText(
        frame, "Khong phat hien khuon mat",
        (w // 2 - 180, h // 2),
        FONT, 0.8, (0, 140, 255), 2
    )
    return frame


print("Cac ham ve UI da san sang.")

Cac ham ve UI da san sang.


---

## Cell 11 - Module dang ky diem danh

Module nay xu ly viec luu diem danh ra CSV khi mot nguoi duoc nhan dien thanh cong.

**Co che chong trung:** Moi nguoi chi duoc ghi nhan mot lan trong moi phien lam viec (session). Neu cung mot nguoi xuat hien nhieu lan, chi ghi lan dau tien.

**Format CSV output:**
```
Name,Time,Date
An,08:32:15,2024-01-15
Binh,08:35:42,2024-01-15
```

In [26]:
# ============================================================
# MODULE DIEM DANH
# ============================================================

class AttendanceLogger:
    """
    Quan ly viec ghi nhan diem danh ra file CSV.
    Co co che chong ghi trung trong cung mot phien.
    """
    
    def __init__(self, csv_path: Path):
        self.csv_path = csv_path
        self.logged_today: set = set()  # Ten nguoi da ghi trong phien nay
        self._ensure_csv_header()
    
    def _ensure_csv_header(self):
        """Tao file CSV voi header neu chua ton tai."""
        if not self.csv_path.exists():
            with open(self.csv_path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(["Name", "Time", "Date"])
            print(f"Da tao file diem danh: {self.csv_path}")
    
    def log(self, name: str) -> bool:
        """
        Ghi nhan diem danh cho mot nguoi.
        
        Args:
            name: Ten nguoi can ghi nhan
        
        Returns:
            True neu ghi thanh cong, False neu da ghi roi trong phien nay
        """
        if name == "Unknown" or name in self.logged_today:
            return False
        
        now = datetime.now()
        time_str = now.strftime("%H:%M:%S")
        date_str = now.strftime("%Y-%m-%d")
        
        try:
            with open(self.csv_path, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow([name, time_str, date_str])
            
            self.logged_today.add(name)
            print(f"[Diem danh] {name} - {time_str} {date_str}")
            return True
        except IOError as e:
            print(f"Loi ghi file diem danh: {e}")
            return False
    
    def reset_session(self):
        """Reset danh sach da diem danh (bat dau phien moi)."""
        self.logged_today.clear()
        print("Da reset phien diem danh.")
    
    def show_today_attendance(self):
        """In danh sach da diem danh trong phien hien tai."""
        if self.logged_today:
            print(f"Da diem danh trong phien nay: {', '.join(sorted(self.logged_today))}")
        else:
            print("Chua co ai diem danh trong phien nay.")


# Khoi tao attendance logger
attendance_logger = AttendanceLogger(ATTENDANCE_FILE)
print("Attendance logger da san sang.")

Da tao file diem danh: attendance_log.csv
Attendance logger da san sang.


---

## Cell 12 - Module dang ky khuon mat moi qua webcam

Cho phep dang ky nguoi moi truc tiep trong khi webcam dang chay bang cach nhan phim `S`.

**Luong xu ly:**
1. Nguoi dung nhan `S`
2. Hien thi cua so nhap ten
3. Chup anh khuon mat hien tai
4. Trich xuat embedding
5. Them vao database va luu anh

**Luu y:** Vi OpenCV khong ho tro popup input box tot, ta dung cv2.waitKey de bat phim va in prompt ra terminal.

In [27]:
# ============================================================
# MODULE DANG KY KHUON MAT MOI
# ============================================================

def register_new_face(
    frame: np.ndarray,
    database: dict,
    dataset_dir: Path
) -> dict:
    """
    Dang ky khuon mat moi tu frame hien tai.
    
    Args:
        frame: Frame hien tai tu webcam
        database: Database hien tai (se duoc cap nhat)
        dataset_dir: Thu muc luu anh
    
    Returns:
        Database da duoc cap nhat
    """
    print("\n--- DANG KY NGUOI MOI ---")
    
    # Detect khuon mat trong frame hien tai
    face = extract_face(frame)
    
    if face is None:
        print("Khong phat hien khuon mat trong frame. Thu lai.")
        return database
    
    # Nhap ten (tu terminal vi OpenCV khong co GUI input)
    print("Nhin thang vao camera va giu nguyen mat trong 3 giay...")
    time.sleep(3)
    
    # Chup lai anh sau khi cho
    print("Nhap ten nguoi (khong dau, khong khoang trang dau cuoi):")
    name = input("Ten: ").strip()
    
    if not name:
        print("Ten khong hop le. Huy dang ky.")
        return database
    
    # Sanitize ten (tranh ky tu dac biet trong ten file)
    safe_name = "".join(c for c in name if c.isalnum() or c in (' ', '-', '_')).strip()
    if not safe_name:
        print("Ten sau khi xu ly bi rong. Huy dang ky.")
        return database
    
    # Tao thu muc luu anh
    person_dir = dataset_dir / safe_name
    person_dir.mkdir(exist_ok=True)
    
    # Dem so anh hien co de dat ten file moi
    existing_imgs = list(person_dir.glob('*.jpg'))
    img_index = len(existing_imgs) + 1
    
    # Luu anh khuon mat
    save_path = person_dir / f"{img_index}.jpg"
    face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(save_path), face_bgr)
    print(f"Da luu anh: {save_path}")
    
    # Trich xuat embedding
    try:
        new_embedding = get_embedding(face)
    except Exception as e:
        print(f"Loi trich xuat embedding: {e}")
        return database
    
    # Them vao database
    if safe_name in database:
        # Neu nguoi da ton tai, tinh lai trung binh
        old_emb = database[safe_name]
        database[safe_name] = (old_emb + new_embedding) / 2.0
        print(f"Da cap nhat embedding cho: {safe_name}")
    else:
        database[safe_name] = new_embedding
        print(f"Da them nguoi moi: {safe_name}")
    
    print(f"Database hien tai: {list(database.keys())}")
    print("--- KET THUC DANG KY ---\n")
    
    return database


print("Module dang ky khuon mat moi da san sang.")
print("Su dung: Nhan phim 'S' trong khi chay webcam de dang ky.")

Module dang ky khuon mat moi da san sang.
Su dung: Nhan phim 'S' trong khi chay webcam de dang ky.


---

## Cell 13 - Cau hinh toi uu hieu nang

Truoc khi chay webcam loop, thiet lap cac thong so de can bang giua do chinh xac va toc do.

**Cac ky thuat toi uu su dung:**

1. **Skip frame khi detect:** MTCNN ton nhieu thoi gian. Neu detect moi frame -> FPS thap. Giai phap: Chi detect moi N frame, cac frame con lai dung lai ket qua cu.

2. **Resize frame truoc khi detect:** Giam resolution truoc khi dua vao MTCNN giup detect nhanh hon. Sau do quy ve toa do goc khi ve box.

3. **Giam resolution webcam:** Set output cam xuong 640x480 hay 480x360 thay vi Full HD.

4. **Caching:** Neu embedding cua frame truoc da tinh roi, dung lai thay vi tinh moi.

**Khuyen nghi theo cap do may tinh:**
- May manh (RAM >=8GB, co GPU): `DETECT_EVERY_N = 1`, `FRAME_WIDTH = 1280`
- May trung binh (RAM 8GB, CPU): `DETECT_EVERY_N = 2`, `FRAME_WIDTH = 640`
- May yeu (RAM 4GB, CPU): `DETECT_EVERY_N = 5`, `FRAME_WIDTH = 480`

In [28]:
# ============================================================
# CAU HINH TOAN CUC (CHINH O DAY)
# ============================================================

class Config:
    """
    Tap trung toan bo cau hinh vao mot class.
    De dang thay doi ma khong can tim khap noi trong code.
    """
    
    # --- Webcam ---
    CAMERA_INDEX: int   = 0      # 0 = cam chinh, 1 = cam ngoai
    FRAME_WIDTH: int    = 640    # Do rong frame output (pixel)
    FRAME_HEIGHT: int   = 480    # Do cao frame output (pixel)
    
    # --- Nhan dien ---
    SIMILARITY_THRESHOLD: float = 0.7   # Nguong nhan dien (0-1)
    DETECT_EVERY_N: int         = 2     # Chi detect khuon mat moi N frame
    DETECT_SCALE: float         = 0.5   # Thu nho frame xuong khi detect (tang toc)
    MIN_FACE_CONFIDENCE: float  = 0.9   # Do tin cay MTCNN toi thieu
    
    # --- Diem danh ---
    ENABLE_ATTENDANCE: bool     = True  # Co ghi diem danh khong
    
    # --- May yeu ---
    LOW_RAM_MODE: bool          = False # True: giam them resolution va skip frame

    @classmethod
    def apply_low_ram_mode(cls):
        """Ap dung cau hinh cho may yeu RAM < 4GB."""
        cls.FRAME_WIDTH     = 480
        cls.FRAME_HEIGHT    = 360
        cls.DETECT_EVERY_N  = 5
        cls.DETECT_SCALE    = 0.4
        cls.LOW_RAM_MODE    = True
        print("Da kich hoat che do may yeu (Low RAM Mode).")
    
    @classmethod
    def print_config(cls):
        """In ra cau hinh hien tai de kiem tra."""
        print("Cau hinh hien tai:")
        print(f"  Camera index        : {cls.CAMERA_INDEX}")
        print(f"  Frame size          : {cls.FRAME_WIDTH}x{cls.FRAME_HEIGHT}")
        print(f"  Similarity threshold: {cls.SIMILARITY_THRESHOLD}")
        print(f"  Detect every N frame: {cls.DETECT_EVERY_N}")
        print(f"  Detect scale        : {cls.DETECT_SCALE}")
        print(f"  Min face confidence : {cls.MIN_FACE_CONFIDENCE}")
        print(f"  Attendance mode     : {cls.ENABLE_ATTENDANCE}")
        print(f"  Low RAM mode        : {cls.LOW_RAM_MODE}")


# --- Uncomment dong nay neu may yeu ---
# Config.apply_low_ram_mode()

Config.print_config()

Cau hinh hien tai:
  Camera index        : 0
  Frame size          : 640x480
  Similarity threshold: 0.7
  Detect every N frame: 2
  Detect scale        : 0.5
  Min face confidence : 0.9
  Attendance mode     : True
  Low RAM mode        : False


---

## Cell 14 - Vong lap nhan dien thoi gian thuc (MAIN LOOP)

Day la cell chinh chay toan bo he thong nhan dien realtime.

**Logic cua vong lap:**
```
Bat dau
  |
  v
Doc frame tu webcam
  |
  v
N chia het DETECT_EVERY_N?
  |-- Co -> Resize frame nho -> MTCNN detect -> Lay bounding box moi
  |-- Khong -> Dung lai bounding box cu
  |
  v
Co khuon mat?
  |-- Khong -> Hien 'Khong phat hien mat', tiep tuc vong lap
  |-- Co -> Crop mat -> FaceNet embedding -> So sanh database
  |
  v
Score > threshold?
  |-- Co -> Hien ten + khung xanh + ghi diem danh
  |-- Khong -> Hien Unknown + khung do
  |
  v
Nhan phim?
  |-- Q -> Thoat
  |-- S -> Dang ky mat moi
  |-- Khac -> Tiep tuc
  |
  v
Hien thi frame -> Quay lai dau
```

**Nhan phim de dieu khien:**
- `Q` hoac `q`: Thoat khoi vong lap
- `S` hoac `s`: Dang ky khuon mat moi

In [29]:
# ============================================================
# REALTIME FACE RECOGNITION - MAIN LOOP
# ============================================================

def run_realtime_recognition(
    database: dict,
    config: Config,
    attendance_logger: AttendanceLogger
) -> dict:
    """
    Vong lap chinh nhan dien khuon mat thoi gian thuc.
    
    Args:
        database: Face database {ten: embedding}
        config: Cau hinh he thong
        attendance_logger: Module ghi diem danh
    
    Returns:
        Database co the da duoc cap nhat (neu dang ky nguoi moi)
    """
    # Mo webcam
    cap = cv2.VideoCapture(config.CAMERA_INDEX)
    
    if not cap.isOpened():
        print("Khong the mo webcam. Kiem tra lai camera.")
        return database
    
    # Thiet lap webcam
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, config.FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, config.FRAME_HEIGHT)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    print("Webcam da san sang. Nhan 'Q' de thoat, 'S' de dang ky nguoi moi.")
    print("-" * 50)
    
    # Bien trang thai
    frame_count     = 0
    fps             = 0.0
    fps_timer       = time.time()
    fps_frame_count = 0
    
    # Cache ket qua detect (de dung lai giua cac frame)
    cached_boxes    = []         # List cac box da detect
    cached_results  = []         # List (ten, score) tuong ung
    
    try:
        while True:
            # Doc frame
            ret, frame = cap.read()
            
            if not ret or frame is None:
                print("Khong doc duoc frame. Ket noi camera co van de.")
                break
            
            frame_count += 1
            fps_frame_count += 1
            
            # Tinh FPS thuc te (cap nhat moi 0.5 giay)
            elapsed = time.time() - fps_timer
            if elapsed >= 0.5:
                fps = fps_frame_count / elapsed
                fps_timer = time.time()
                fps_frame_count = 0
            
            # BUOC 1: Detect khuon mat (khong phai moi frame)
            should_detect = (frame_count % config.DETECT_EVERY_N == 0)
            
            if should_detect:
                # Thu nho frame de detect nhanh hon
                detect_frame = cv2.resize(
                    frame,
                    None,
                    fx=config.DETECT_SCALE,
                    fy=config.DETECT_SCALE
                )
                
                # Detect tat ca khuon mat
                detected = extract_all_faces_with_boxes(
                    detect_frame,
                    min_confidence=config.MIN_FACE_CONFIDENCE
                )
                
                # Quy doi toa do box ve kich thuoc frame goc
                scale_inv = 1.0 / config.DETECT_SCALE
                
                new_boxes = []
                new_results = []
                
                for face_array, box in detected:
                    # Quy doi toa do
                    orig_box = {
                        'x1': int(box['x1'] * scale_inv),
                        'y1': int(box['y1'] * scale_inv),
                        'x2': int(box['x2'] * scale_inv),
                        'y2': int(box['y2'] * scale_inv),
                        'confidence': box['confidence']
                    }
                    
                    # BUOC 2: Trich xuat embedding tu anh khuon mat goc (khong resize)
                    # Crop lai tu frame goc de embedding chinh xac hon
                    ih, iw = frame.shape[:2]
                    fx1 = max(0, orig_box['x1'])
                    fy1 = max(0, orig_box['y1'])
                    fx2 = min(iw, orig_box['x2'])
                    fy2 = min(ih, orig_box['y2'])
                    
                    if fx2 > fx1 and fy2 > fy1:
                        face_crop = frame[fy1:fy2, fx1:fx2]
                        face_crop_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
                        face_resized = np.array(
                            Image.fromarray(face_crop_rgb).resize((FACE_SIZE, FACE_SIZE))
                        )
                        
                        # BUOC 3: So sanh voi database
                        try:
                            emb = get_embedding(face_resized)
                            name, score = compare_with_database(
                                emb, database, config.SIMILARITY_THRESHOLD
                            )
                        except Exception:
                            name, score = "Unknown", 0.0
                        
                        new_boxes.append(orig_box)
                        new_results.append((name, score))
                        
                        # BUOC 4: Ghi diem danh neu can
                        if config.ENABLE_ATTENDANCE and name != "Unknown":
                            attendance_logger.log(name)
                
                # Cap nhat cache
                cached_boxes   = new_boxes
                cached_results = new_results
            
            # BUOC 5: Ve ket qua len frame (dung ket qua cache)
            display_frame = frame.copy()
            
            if cached_boxes:
                for box, (name, score) in zip(cached_boxes, cached_results):
                    display_frame = draw_face_box(display_frame, box, name, score)
            else:
                display_frame = draw_no_face_indicator(display_frame)
            
            # Ve FPS va huong dan
            mode_text = "Attendance ON" if config.ENABLE_ATTENDANCE else ""
            display_frame = draw_fps_and_info(display_frame, fps, mode_text)
            
            # BUOC 6: Hien thi
            cv2.imshow("Face Recognition - FaceNet + MTCNN", display_frame)
            
            # BUOC 7: Bat phim
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q') or key == ord('Q'):
                print("Da nhan Q. Dang thoat...")
                break
            
            elif key == ord('s') or key == ord('S'):
                # Dang ky khuon mat moi
                database = register_new_face(frame, database, DATASET_DIR)
                # Reset cache sau khi them nguoi moi
                cached_boxes   = []
                cached_results = []
    
    except KeyboardInterrupt:
        print("\nDa dung boi nguoi dung (Ctrl+C).")
    
    finally:
        # BUOC 8: Giai phong tai nguyen
        cap.release()
        cv2.destroyAllWindows()
        
        # Xu ly them tren Jupyter (tranh cua so bi treo)
        for _ in range(5):
            cv2.waitKey(1)
        
        print(f"\nTong so frame da xu ly: {frame_count}")
        print(f"FPS trung binh cuoi: {fps:.1f}")
        attendance_logger.show_today_attendance()
        print("Ket thuc nhan dien.")
    
    return database


print("Ham run_realtime_recognition da san sang.")
print("Chay Cell tiep theo de bat dau nhan dien.")

Ham run_realtime_recognition da san sang.
Chay Cell tiep theo de bat dau nhan dien.


---

## Cell 15 - KHOI CHAY HE THONG

Day la cell duy nhat can chay de khoi dong toan bo he thong.

**Truoc khi chay hay kiem tra:**
1. Camera hoat dong (Cell 6 phai bao OK)
2. Database co it nhat 1 nguoi (Cell 9 hien ro so nguoi)
3. Cau hinh da dung voi may cua ban (Cell 13)

**Cua so hien thi:**
- Mot cua so OpenCV se xuat hien
- Nhin truc tiep vao camera
- Cua so nay can co focus (click vao no) de bat phim Q, S

**Luu y quan trong:**
- Neu cua so bi treo, chay lai toan bo kernel (Kernel -> Restart)
- Neu FPS qua thap (< 5), tang `DETECT_EVERY_N` len 5 hoac 10
- Tren Jupyter Lab/Notebook: Cu so OpenCV se pop up rieng, KHONG hien trong notebook

In [30]:
# ============================================================
# KHOI CHAY - CHAY CELL NAY DE BAT DAU
# ============================================================

print("Kiem tra truoc khi khoi chay:")
print(f"  Camera index : {Config.CAMERA_INDEX}")
print(f"  Database     : {len(database)} nguoi - {list(database.keys())}")
print(f"  Threshold    : {Config.SIMILARITY_THRESHOLD}")
print(f"  Detect every : {Config.DETECT_EVERY_N} frame")

if not database:
    print("\nCANH BAO: Database dang trong!")
    print("He thong van chay nhung moi nguoi se hien 'Unknown'.")
    print("Hay nhan phim S de dang ky nguoi moi khi webcam dang chay.")

print("\nDang khoi dong webcam...")
print("Su dung cua so OpenCV -> Nhan Q de thoat, S de dang ky nguoi moi")
print("-" * 60)

# Khoi chay he thong
database = run_realtime_recognition(
    database=database,
    config=Config,
    attendance_logger=attendance_logger
)

Kiem tra truoc khi khoi chay:
  Camera index : 0
  Database     : 1 nguoi - ['Nghiem Duc Thuan']
  Threshold    : 0.7
  Detect every : 2 frame

Dang khoi dong webcam...
Su dung cua so OpenCV -> Nhan Q de thoat, S de dang ky nguoi moi
------------------------------------------------------------


Webcam da san sang. Nhan 'Q' de thoat, 'S' de dang ky nguoi moi.
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
[Diem danh] Nghiem Duc Thuan - 20:45:53 2026-04-20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/st

---

## Cell 16 - Xem lai file diem danh

Sau khi chay, doc va hien thi noi dung file diem danh da ghi.

In [31]:
# ============================================================
# XEM FILE DIEM DANH
# ============================================================

import pandas as pd

def show_attendance_log(csv_path: Path):
    """Doc va hien thi file diem danh."""
    if not csv_path.exists():
        print("File diem danh chua duoc tao.")
        return
    
    try:
        df = pd.read_csv(csv_path, encoding='utf-8')
        
        if df.empty:
            print("File diem danh trong (chua co ban ghi nao).")
            return
        
        print(f"File diem danh: {csv_path.absolute()}")
        print(f"Tong so ban ghi: {len(df)}")
        print(f"So nguoi khac nhau: {df['Name'].nunique()}")
        print()
        
        # Hien thi dang dep
        print(df.to_string(index=False))
        
        # Thong ke
        print("\nThong ke theo ngay:")
        if 'Date' in df.columns:
            summary = df.groupby('Date')['Name'].count().reset_index()
            summary.columns = ['Date', 'So nguoi diem danh']
            print(summary.to_string(index=False))
    
    except Exception as e:
        print(f"Loi doc file diem danh: {e}")
        # Fallback: in thu cong
        with open(csv_path, 'r', encoding='utf-8') as f:
            print(f.read())


show_attendance_log(ATTENDANCE_FILE)

File diem danh: d:\Computer_Vision\computer-vision\notebooks\attendance_log.csv
Tong so ban ghi: 1
So nguoi khac nhau: 1

            Name     Time       Date
Nghiem Duc Thuan 20:45:53 2026-04-20

Thong ke theo ngay:
      Date  So nguoi diem danh
2026-04-20                   1


---

## Cell 17 - Truc quan hoa embedding (Bonus phan tich)

Su dung PCA (Principal Component Analysis) de giam tu 512 chieu xuong 2 chieu va ve bieu do.

**Muc dich:** Truc quan hoa xem cac embedding cua cung mot nguoi co phan cum voi nhau khong, va embedding cua nguoi khac nhau co tach biet nhau khong.

**Ket qua mong doi:** Moi nguoi se tao thanh mot cum diem rieng biet tren bieu do 2D.

In [32]:
# ============================================================
# TRUC QUAN HOA EMBEDDING SPACE
# ============================================================

from sklearn.decomposition import PCA

def visualize_embeddings(database: dict):
    """
    Truc quan hoa cac embedding vector trong khong gian 2D bang PCA.
    
    Chi co y nghia khi database co tu 2 nguoi tro len.
    """
    if len(database) < 2:
        print("Can it nhat 2 nguoi trong database de truc quan hoa.")
        return
    
    names = list(database.keys())
    vectors = np.array([database[name] for name in names])
    
    # PCA giam xuong 2 chieu
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(vectors)
    
    explained = pca.explained_variance_ratio_
    
    # Ve bieu do
    fig, ax = plt.subplots(figsize=(8, 6))
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(names)))
    
    for i, (name, color) in enumerate(zip(names, colors)):
        ax.scatter(
            reduced[i, 0], reduced[i, 1],
            color=color, s=200, zorder=5,
            label=name
        )
        ax.annotate(
            name,
            (reduced[i, 0], reduced[i, 1]),
            textcoords="offset points",
            xytext=(10, 5),
            fontsize=12
        )
    
    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}% variance)")
    ax.set_title("Face Embedding Space (PCA 2D)\nMoi diem la 1 nguoi trong database")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("embedding_visualization.png", dpi=120, bbox_inches='tight')
    plt.show()
    
    print(f"\nTong phuong sai giai thich boi 2 PC dau: {sum(explained)*100:.1f}%")
    print(f"Da luu bieu do: embedding_visualization.png")


# Also compute pairwise similarity matrix
def show_similarity_matrix(database: dict):
    """
    Hien thi ma tran cosine similarity giua cac nguoi trong database.
    Dieu nay giup chon threshold phu hop.
    """
    if len(database) < 2:
        print("Can it nhat 2 nguoi de tinh ma tran similarity.")
        return
    
    names = list(database.keys())
    vectors = np.array([database[name] for name in names])
    
    sim_matrix = cosine_similarity(vectors)
    
    fig, ax = plt.subplots(figsize=(max(5, len(names)), max(4, len(names)-1)))
    
    im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
    
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_yticklabels(names)
    
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{sim_matrix[i,j]:.3f}",
                   ha='center', va='center', fontsize=11,
                   color='black' if sim_matrix[i,j] < 0.8 else 'white')
    
    plt.colorbar(im, ax=ax, label='Cosine Similarity')
    ax.set_title("Cosine Similarity Matrix giua cac nguoi trong Database")
    
    plt.tight_layout()
    plt.savefig("similarity_matrix.png", dpi=120, bbox_inches='tight')
    plt.show()
    
    print("\nNhan xet:")
    print("  Duong cheo chinh (1.0): similarity cua nguoi voi chinh ho")
    print("  Cac o con lai: similarity giua nguoi khac nhau")
    print("  Gia tri thap o o khac duong cheo = database tot, nguoi khac nhau ro rang")


if database:
    visualize_embeddings(database)
    show_similarity_matrix(database)
else:
    print("Database trong. Chay Cell 9 de xay dung database truoc.")

Can it nhat 2 nguoi trong database de truc quan hoa.
Can it nhat 2 nguoi de tinh ma tran similarity.


---

## Cell 18 - Xu ly cac loi thuong gap

Phieu tham khao tra cuu nhanh khi gap loi.

In [33]:
# ============================================================
# TROUBLESHOOTING GUIDE
# ============================================================

TROUBLESHOOTING = """
============================================================
HUONG DAN XU LY LOI THUONG GAP
============================================================

LOI 1: cannot open webcam / Camera not found
  Nguyen nhan:
    - Khong co camera hoac driver chua cai
    - App khac dang dung camera (Zoom, Teams)
    - Camera index sai
  Cach su'a:
    - Dong tat ca app co dung camera
    - Thu doi CAMERA_INDEX tu 0 sang 1 hoac 2
    - Kiem tra Device Manager (Windows)

LOI 2: No face detected
  Nguyen nhan:
    - Anh toi, anh sau, mat bi che khuat
    - Khuon mat qua xa camera
    - MIN_FACE_CONFIDENCE dang qua cao
  Cach su'a:
    - Them anh sang hoac ngoi sat camera hon
    - Giam MIN_FACE_CONFIDENCE tu 0.9 xuong 0.7
    - Thay doi goc nhin khuon mat

LOI 3: tensorflow DLL load failed (Windows)
  Nguyen nhan:
    - Thieu Microsoft Visual C++ Redistributable
  Cach su'a:
    - Tai va cai: Microsoft Visual C++ 2019 Redistributable x64
    - Link: https://aka.ms/vs/17/release/vc_redist.x64.exe

LOI 4: MTCNN import error / ModuleNotFoundError
  Cach su'a:
    - pip install mtcnn --upgrade
    - Kiem tra Python environment dang dung
    - Tren Windows: Thu pip install mtcnn==0.1.1

LOI 5: FPS thap (< 5 FPS)
  Nguyen nhan:
    - May yeu, khong co GPU
    - DETECT_EVERY_N qua thap
  Cach su'a:
    - Tang DETECT_EVERY_N len 5 hoac 10
    - Giam FRAME_WIDTH xuong 480 hoac 320
    - Goi Config.apply_low_ram_mode()

LOI 6: Tat ca hien Unknown du mat dung
  Nguyen nhan:
    - SIMILARITY_THRESHOLD qua cao
    - Database chua duoc xay dung
    - Anh trong dataset chat luong kem
  Cach su'a:
    - Giam SIMILARITY_THRESHOLD tu 0.7 xuong 0.6
    - Chay lai Cell 9 de rebuild database
    - Them anh chat luong cao vao dataset

LOI 7: Jupyter kernel chet sau khi chay webcam
  Nguyen nhan:
    - OpenCV window bi treo khi dong doi biet
  Cach su'a:
    - Luon nhan Q de thoat thay vi dong cua so X
    - Neu kernel chet: Kernel -> Restart, chay lai tu Cell 4

============================================================
"""

print(TROUBLESHOOTING)


HUONG DAN XU LY LOI THUONG GAP

LOI 1: cannot open webcam / Camera not found
  Nguyen nhan:
    - Khong co camera hoac driver chua cai
    - App khac dang dung camera (Zoom, Teams)
    - Camera index sai
  Cach su'a:
    - Dong tat ca app co dung camera
    - Thu doi CAMERA_INDEX tu 0 sang 1 hoac 2
    - Kiem tra Device Manager (Windows)

LOI 2: No face detected
  Nguyen nhan:
    - Anh toi, anh sau, mat bi che khuat
    - Khuon mat qua xa camera
    - MIN_FACE_CONFIDENCE dang qua cao
  Cach su'a:
    - Them anh sang hoac ngoi sat camera hon
    - Giam MIN_FACE_CONFIDENCE tu 0.9 xuong 0.7
    - Thay doi goc nhin khuon mat

LOI 3: tensorflow DLL load failed (Windows)
  Nguyen nhan:
    - Thieu Microsoft Visual C++ Redistributable
  Cach su'a:
    - Tai va cai: Microsoft Visual C++ 2019 Redistributable x64
    - Link: https://aka.ms/vs/17/release/vc_redist.x64.exe

LOI 4: MTCNN import error / ModuleNotFoundError
  Cach su'a:
    - pip install mtcnn --upgrade
    - Kiem tra Python enviro

---

## Cell 19 - Ket luan

### Tong ket bai lab

Trong bai lab nay chung ta da xay dung thanh cong he thong nhan dien khuon mat thoi gian thuc voi cac thanh phan:

**Nhung gi da lam duoc:**

- Xay dung pipeline day du: Webcam -> MTCNN detect -> FaceNet embed -> Cosine similarity -> Hien thi
- Toi uu FPS bang cach skip frame va resize truoc khi detect
- Xu ly nhieu nguoi trong cung mot frame
- He thong dang ky khuon mat moi truc tiep qua webcam
- Module diem danh tu dong ghi ra CSV
- Truc quan hoa embedding space de hieu model

---

**Uu diem cua FaceNet:**

- Khong can retrain khi them nguoi moi, chi can them embedding vao database
- Chay tot tren CPU (khong bat buoc phai co GPU)
- Do chinh xac cao tren nhiều tap benchmark quoc te
- Embedding co the luu vao file, khong can giu anh goc

---

**Nhuoc diem va gioi han:**

- Anh sang yeu lam giam do chinh xac cua ca MTCNN va FaceNet
- Khuon mat deo kinh dam, deo mask co the khong detect duoc
- FPS con thap tren may yeu do MTCNN ton nhieu tinh toan
- Voi database lon (> 1000 nguoi), can dung ANN (Approximate Nearest Neighbor) thay vi so sanh truc tiep

---

**Tam quan trong cua threshold:**

- Threshold = 0.7 la gia tri trung binh, can chinh tuy theo dieu kien anh sang va chat luong dataset
- Threshold cao -> it nhan dien sai nhung bo sot nhieu (precision cao, recall thap)
- Threshold thap -> nhan dien duoc nhieu hon nhung co the nhan sai (precision thap, recall cao)
- Nen thu nghiem tren tap kiem thu rieng truoc khi chon threshold chinh thuc

---

### Huong phat trien tiep theo

**Nang cap mo hinh:**
- **ArcFace / InsightFace**: Do chinh xac cao hon FaceNet, dac biet voi khuon mat nghieng
- **DeepFace**: Thu vien wrapper ho tro nhieu backend (VGG-Face, Facenet, ArcFace)
- **AdaFace**: Hieu nang tot voi anh chat luong thap (toi, mo)

**Mo rong ung dung:**
- Deploy thanh Flask Web App cho phep nhan dien qua trinh duyet
- Xay dung API REST de tich hop vao cac he thong khac
- Trien khai tren Raspberry Pi de lam Smart Door Lock
- Tich hop voi CCTV camera IP qua RTSP stream
- Xay dung giao dien quan tri database bang web (them/xoa/sua nguoi)

**Toi uu hieu nang:**
- Su dung FAISS (Facebook AI Similarity Search) de tim kiem embedding nhanh hon trong database lon
- Convert model sang TensorRT hoac ONNX de chay nhanh hon
- Dung threading/multiprocessing tach rieng viec doc frame va xu ly

---

*Bai lab hoan thanh. Cam on thay/co huong dan!*

In [34]:
# ============================================================
# TONG KET THONG TIN HE THONG
# ============================================================

import platform

print("========== THONG TIN HE THONG ==========")
print(f"Python version  : {sys.version.split()[0]}")
print(f"OS              : {platform.system()} {platform.release()}")
print(f"OpenCV version  : {cv2.__version__}")
print(f"NumPy version   : {np.__version__}")
print()

print("========== THONG TIN DATABASE ==========")
print(f"So nguoi trong database : {len(database)}")
if database:
    sample_name = next(iter(database))
    print(f"Kich thuoc embedding    : {database[sample_name].shape}")
    print(f"Danh sach nguoi         : {', '.join(database.keys())}")
print()

print("========== CAU HINH HIEN TAI ==========")
Config.print_config()
print()

print("========== TONG KET ==========")
print("He thong nhan dien khuon mat FaceNet + MTCNN")
print("Da xay dung thanh cong cac module:")
print("  [OK] MTCNN face detection")
print("  [OK] FaceNet face embedding")
print("  [OK] Cosine similarity matching")
print("  [OK] Realtime webcam loop co toi uu FPS")
print("  [OK] Face registration (nhan phim S)")
print("  [OK] Attendance logging ra CSV")
print("  [OK] Embedding visualization (PCA)")
print("  [OK] Similarity matrix heatmap")
print("  [OK] Troubleshooting guide")
print("  [OK] Low RAM mode cho may yeu")

========== THONG TIN HE THONG ==========
Python version  : 3.12.10
OS              : Windows 11
OpenCV version  : 4.8.1
NumPy version   : 1.26.4

========== THONG TIN DATABASE ==========
So nguoi trong database : 1
Kich thuoc embedding    : (512,)
Danh sach nguoi         : Nghiem Duc Thuan

========== CAU HINH HIEN TAI ==========
Cau hinh hien tai:
  Camera index        : 0
  Frame size          : 640x480
  Similarity threshold: 0.7
  Detect every N frame: 2
  Detect scale        : 0.5
  Min face confidence : 0.9
  Attendance mode     : True
  Low RAM mode        : False

========== TONG KET ==========
He thong nhan dien khuon mat FaceNet + MTCNN
Da xay dung thanh cong cac module:
  [OK] MTCNN face detection
  [OK] FaceNet face embedding
  [OK] Cosine similarity matching
  [OK] Realtime webcam loop co toi uu FPS
  [OK] Face registration (nhan phim S)
  [OK] Attendance logging ra CSV
  [OK] Embedding visualization (PCA)
  [OK] Similarity matrix heatmap
  [OK] Troubleshooting guide
  [OK